# Infomation

**Hello, this is the method we used to solve the problem of this tournament and get the second place on the leaderboard.**

We do feature processing by identifying information including:
* - Identifying information about calendar holidays according to the countries containing the warehouses.
* - Processing the names of holidays.
* - Splitting the warehouse information into columns to form a sparse table.
* - Using onehotcoder and TFIDF for processing.

We use Optina for finetuning hyperparameter for LGBM. We using a single LGBM

**Note:** The solution that achieved first place in public did not perform as well as our previous solution when evaluated in private data. We share the best solution we came up with.

<center><img src="https://www.indexventures.com/media/images/Rohlik_1800x1200_cop.2e16d0ba.format-jpeg.fill-1800x1200.jpg" style="width:1000px;height:600px;"></center>

**Please don't forget upvote if you like my works** 

# 1. Import modules and load data

#### 1. Import library

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import re
import string
from datetime import datetime, timedelta
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMRegressor

In [ ]:
!mkdir /kaggle/working/checkpoint

In [ ]:
data_path = '/kaggle/input/rohlik-orders-forecasting-challenge'
checkpoint_path = '/kaggle/working/checkpoint'

df = pd.read_csv(f'{data_path}/train.csv')
test = pd.read_csv(f'{data_path}/test.csv')

# Read calendar dataset
df_cld = pd.read_csv(f'{data_path}/train_calendar.csv')
test_cld = pd.read_csv(f'{data_path}/test_calendar.csv')
df_cld['date'] = pd.to_datetime(df_cld['date'], errors='coerce')
test_cld['date'] = pd.to_datetime(test_cld['date'], errors='coerce')

df['date'] = pd.to_datetime(df['date'], errors='coerce')
test['date'] = pd.to_datetime(test['date'], errors='coerce')

# Dictionary for renaming specific holiday names
rename_dict = {
    "Memorial Day for the Victims of the Holocaust": "Victims of the Holocaust",
    "Memorial Day for the Victims of the Communist Dictatorships": "Victims of the Communist",
    "Den vzniku samostatneho ceskoslovenskeho statu": "Den vzniku"
}

# Replace the values in the 'holiday_name' column based on the dictionary
df['holiday_name'] = df['holiday_name'].replace(rename_dict)
test['holiday_name'] = test['holiday_name'].replace(rename_dict)

testids = test['id']
df_ori = df.copy()
test_ori = test.copy()
df.head()

#### 2.Data representation

In [ ]:
df.info()

In [ ]:
df.describe().T

# 2. Dataset Analyst

Displays data on order levels by warehouse

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# Plot histograms
for i, wh in enumerate(df['warehouse'].unique()):
    ax = axes[i]
    ax.hist(df[df['warehouse'] == wh]['orders'], bins=300)
    ax.set_title(f'Warehouse: {wh}')
    ax.set_xlabel('Orders')
    ax.set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Until display predictions for by warehouse, Show data over the entire period with holidays and with Good Friday holiday

In [ ]:
def display_predictions(target_column, years:list, plot_continue:bool, train_df: pd.DataFrame, force_holiday:str, use_submission:bool, submission_df: pd.DataFrame=None) -> None:
    """
    Plot full timeline including predictions.

    target_column: Column name for the target variable.
    years: years to plot.
    plot_continue: ignore, have some bug now.
    train_df: Train data.
    force_holiday: Zoom to date around this day for easy to observe.
    use_submission: Include was predict orders in plot.
    submission_df: You last submission dataframe. DataFrame containing the predictions. Must also include warehouse names.
    """
    if use_submission:
        submission_df = submission_df.rename(columns={"id": "warehouse"})
        submission_df['date'] = submission_df['warehouse'].str.split('_').str[2]
        submission_df['warehouse'] = submission_df['warehouse'].str.split('_').str[:2].str.join('_')
        submission_df['date'] = pd.to_datetime(submission_df['date'], errors='coerce')

    train_df['date'] = pd.to_datetime(train_df['date'], errors='coerce')

    if force_holiday:
        plot_continue=True
        for year in years:
            plt.figure(figsize=(15, 6))
            for wh in train_df["warehouse"].unique():
                wh_train = train_df[(train_df["warehouse"] == wh) & (train_df['date'].dt.year==year)][["warehouse", target_column, "date", "holiday", "holiday_name"]].set_index("date")
                if use_submission:
                    wh_submission = submission_df[submission_df["warehouse"] == wh][["warehouse", target_column, "date"]].set_index("date")

                if force_holiday:
                    # Get the dates of the force_holiday
                    holiday_dates = wh_train[wh_train['holiday_name'] == force_holiday].index

                    if holiday_dates.empty:
                        continue

                    # Consider only the first occurrence of the force_holiday
                    holiday_date = holiday_dates[0]
                    start_date = holiday_date - pd.Timedelta(days=7)
                    end_date = holiday_date + pd.Timedelta(days=3)

                    wh_train = wh_train[(wh_train.index >= start_date) & (wh_train.index <= end_date)]
                    if use_submission:
                        wh_submission = wh_submission[(wh_submission.index >= start_date) & (wh_submission.index <= end_date)]

                if plot_continue:
                    # Plot all years in one line
                    plt.plot(wh_train.index, wh_train[target_column], label=wh, linewidth=1)
                    if use_submission:
                        plt.plot(wh_submission.index, wh_submission[target_column], label='Predictions', color='red', linewidth=0.8)
                else:
                    # Plot each year as one line
                    for year in years:
                        wh_train_year = wh_train[wh_train.index.year == year]
                        plt.plot(wh_train_year.index, wh_train_year[target_column], label=f'Train Data {year}', linewidth=0.8)
                        if use_submission:
                            wh_submission_year = wh_submission[wh_submission.index.year == year]
                            plt.plot(wh_submission_year.index, wh_submission_year[target_column], label=f'Predictions {year}', linewidth=0.8)

                # Add vertical lines for holidays
                for holiday_date in holiday_dates:
                    plt.axvline(x=holiday_date, linestyle='--', linewidth=0.6, label=f'{force_holiday}_{wh}' if holiday_date == holiday_dates[0] else "")

            plt.title(f"Orders for holiday {year}")
            plt.xlabel("Date")
            plt.ylabel(target_column)
            plt.legend()
            plt.xticks(rotation=45)
            plt.grid(axis='y')
            plt.show()
    else:
        for wh in train_df["warehouse"].unique():
            wh_train = train_df[train_df["warehouse"] == wh][["warehouse", target_column, "date", "holiday", "holiday_name"]].set_index("date")
            if use_submission:
                wh_submission = submission_df[submission_df["warehouse"] == wh][["warehouse", target_column, "date"]].set_index("date")

            plt.figure(figsize=(15, 6))

            if plot_continue:
                # Plot all years in one line
                plt.plot(wh_train.index, wh_train[target_column], label='Train Data', color='blue', linewidth=0.8)
                if use_submission:
                    plt.plot(wh_submission.index, wh_submission[target_column], label='Predictions', color='red', linewidth=0.8)
            else:
                # Plot each year as one line
                for year in years:
                    wh_train_year = wh_train[wh_train.index.year == year]
                    plt.plot(wh_train_year.index, wh_train_year[target_column], label=f'Train Data {year}', linewidth=0.8)
                    if use_submission:
                        wh_submission_year = wh_submission[wh_submission.index.year == year]
                        plt.plot(wh_submission_year.index, wh_submission_year[target_column], label=f'Predictions {year}', linewidth=0.8)

            # Add vertical lines for holidays
            holiday_dates = wh_train[wh_train['holiday'] == 1].index
            for holiday_date in holiday_dates:
                plt.axvline(x=holiday_date, color='green', linestyle='--', linewidth=0.6, label='Holiday' if holiday_date == holiday_dates[0] else "")

            # # Add vertical lines for 'Good Friday'
            # good_friday_dates = wh_train[wh_train['holiday_name'] == 'good friday'].index
            # for good_friday_date in good_friday_dates:
            #     plt.axvline(x=good_friday_date, color='red', linestyle='--', linewidth=0.8, label='Good Friday' if good_friday_date == good_friday_dates[0] else "")

            # # Add vertical lines for all Fridays
            # fridays = wh_train[wh_train.index.dayofweek == 4].index
            # for friday in fridays:
            #     plt.axvline(x=friday, color='purple', linestyle='--', linewidth=0.7, label='Friday' if friday == fridays[0] else "")

            # # Add vertical lines for days before holidays
            # day_before_holiday_dates = (wh_train.index[wh_train['holiday'] == 1] - pd.Timedelta(days=1)).intersection(wh_train.index)
            # for day_before_holiday in day_before_holiday_dates:
            #     plt.axvline(x=day_before_holiday, color='orange', linestyle='--', linewidth=0.7, label='Day before holiday' if day_before_holiday == day_before_holiday_dates[0] else "")

            plt.title(f"Orders for warehouse {wh}")
            plt.xlabel("Date")
            plt.ylabel(target_column)
            plt.legend()
            plt.xticks(rotation=45)
            plt.grid(axis='y')
            plt.show()

In [ ]:
display_predictions(target_column = 'orders',
                    years=[2020,2021,2022,2023,2024],
                    plot_continue=False,
                    # force_holiday='Good Friday', # only show some days around Good Friday
                    force_holiday=None,
                    train_df=df, use_submission=False)

In [ ]:
# View
display_predictions(target_column = 'orders',
                    years=[2021,2022,2023],
                    plot_continue=False,
                    force_holiday='Good Friday', # only show some days around Good Friday
                    # force_holiday=None,
                    train_df=df, use_submission=False)

In [ ]:
# Function to plot mean orders by days in the week for each warehouse
def mean_order(df:pd.DataFrame, warehouse:str, month_list:list) -> None:

    # Filter the DataFrame for select warehouse
    prague_df = df[df['warehouse'] == warehouse]

    prague_df = prague_df[prague_df['date'].dt.month.isin(month_list)]

    # Extract year and day of the week
    prague_df['year'] = prague_df['date'].dt.year
    prague_df['day_of_week'] = prague_df['date'].dt.day_name()

    # Calculate mean orders for each day of the week for each year
    mean_orders_by_day_year = prague_df.groupby(['year', 'day_of_week'])['orders'].mean().unstack()

    teamp_days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    days_order = []
    for day in teamp_days_order:
        if day in mean_orders_by_day_year.columns:
            days_order.append(day)
    mean_orders_by_day_year = mean_orders_by_day_year[days_order]

    plt.figure(figsize=(14, 6))
    for year in mean_orders_by_day_year.index:
        plt.plot(mean_orders_by_day_year.columns, mean_orders_by_day_year.loc[year], marker='o', linestyle='-', label=f'Year {year}')

    plt.title(f'Mean Orders by Day of the Week for {warehouse}')
    plt.xlabel('Day of the Week')
    plt.ylabel('Mean Orders')
    plt.legend()
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    plt.show()

In [ ]:
whs = [
        'Prague_1',
        'Brno_1',
        'Prague_2',
        'Prague_3',
        'Munich_1',
        'Frankfurt_1',
        'Budapest_1'
      ]

# check mean order for day of the week on month 3,4,5
for wh in whs:
    mean_order(df, wh, month_list=[3, 4, 5])

# 3. Data processing

## 1. Fill some new holiday to "train_calendar.csv" and "test_calendar.csv"

In [ ]:
# Holidays get from https://www.holidays-info.com/
# https://www.holidays-info.com/czech-republic/calendar/prague/2024/
czech_holiday = [ # Prague
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020'], 'Easter Day'),#loss
    (['05/12/2024', '05/10/2020', '05/09/2021', '05/08/2022', '05/14/2023'], "Mother Day"), #loss
]
brno_holiday = [ # Brno
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020'], 'Easter Day'),#loss
    (['05/12/2024', '05/10/2020', '05/09/2021', '05/08/2022', '05/14/2023'], "Mother Day"), #loss
]

budapest_holidays = []

# Bavaria - Munich
munich_holidays = [
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021'], 'Holy Saturday'),#loss
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021'], 'Mother Day'),#loss
]

# Hesse - Frankfurt
frank_holidays = [
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021'], 'Holy Saturday'),#loss
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021'], 'Mother Day'),#loss
]


# Generate dates for fixed-date holidays
def generate_dates_for_years(month_day, start_year=2020, end_year=2024):
    dates = []
    for year in range(start_year, end_year + 1):
        date_str = f"{year}-{month_day}"
        dates.append(date_str)
    return dates

def fill_loss_holidays(df_fill, warehouses, holidays):
    df = df_fill.copy()
    # Process each holiday entry
    for item in holidays:
        dates, holiday_name = item

        # If dates is a single date string(fixed holidays), generate the dates for all years
        if isinstance(dates, str):
            month_day = datetime.strptime(dates, '%m/%d/%Y').strftime('%m-%d')
            generated_dates = generate_dates_for_years(month_day)
        else:
            # Convert the dates in the list to the format in the DataFrame
            generated_dates = [datetime.strptime(date, '%m/%d/%Y').strftime('%Y-%m-%d') for date in dates]

        # Update
        for generated_date in generated_dates:
            df.loc[(df['warehouse'].isin(warehouses)) & (df['date'] == generated_date), 'holiday'] = 1
            df.loc[(df['warehouse'].isin(warehouses)) & (df['date'] == generated_date), 'holiday_name'] = holiday_name
    return df

df_cld = df_cld[['warehouse', 'date', 'holiday', 'holiday_name']]
test_cld = test_cld[['warehouse', 'date', 'holiday', 'holiday_name']].reset_index()

# Start fill holidays with above holidays list
df_cld = fill_loss_holidays(df_fill=df_cld, warehouses=['Prague_1', 'Prague_2', 'Prague_3'], holidays=czech_holiday)
test_cld = fill_loss_holidays(df_fill=test_cld, warehouses=['Prague_1', 'Prague_2', 'Prague_3'], holidays=czech_holiday)

df_cld = fill_loss_holidays(df_fill=df_cld, warehouses=['Brno_1'], holidays=brno_holiday)
test_cld = fill_loss_holidays(df_fill=test_cld, warehouses=['Brno_1'], holidays=brno_holiday)

df_cld = fill_loss_holidays(df_fill=df_cld, warehouses=['Munich_1'], holidays=munich_holidays)
test_cld = fill_loss_holidays(df_fill=test_cld, warehouses=['Munich_1'], holidays=munich_holidays)

df_cld = fill_loss_holidays(df_fill=df_cld, warehouses=['Frankfurt_1'], holidays=frank_holidays)
test_cld = fill_loss_holidays(df_fill=test_cld, warehouses=['Frankfurt_1'], holidays=frank_holidays)

df_cld = fill_loss_holidays(df_fill=df_cld, warehouses=['Budapest_1'], holidays=budapest_holidays)
test_cld = fill_loss_holidays(df_fill=test_cld, warehouses=['Budapest_1'], holidays=budapest_holidays)

df_cld = df_cld[df_cld['holiday']==1].reset_index(drop=True)
test_cld = test_cld[test_cld['holiday']==1].reset_index(drop=True)

# Day before Easter Monday is holiday but don't have name,
# we manual fill it by Easter Monday
df_cld = df_cld.fillna('Easter Monday')
test_cld = test_cld.fillna('Easter Monday')

# Sort df for next step
df_cld  = df_cld.sort_values(by=['warehouse', 'date'])
test_cld  = test_cld.sort_values(by=['warehouse', 'date'])

In [ ]:
# Create copy to avoid need re-run from scartch, faster experiment
df_c = df_ori.copy()
test_c = test_ori.copy()

## 2. Fill holiday from calendar to Dataframe

### Main idea:
- In data train, test have column "holiday" with value is 0 and 1
- "fill_calendar2df" will get holiday from calendar.csv(df_cld) to fill for dataframe
- With each holiday, we mark both before and after the holiday as 1 (treat it as a holiday).
- With 2 special holiday('Labour Day' and 'Easter Monday'), we mark 2 days before and  one day after the holiday as 1(Only 2 days before the holiday is weighted at 1.3 instead of 1, I hypothesize that users order more to prepare for the holiday.)

### Example:

#### Input - holiday Easter Monday:

| warehouse | date       | holiday | holiday_name  |
|-----------|------------|---------|---------------|
| A         | 2024-03-28 | 0       | Not           |
| A         | 2024-03-29 | 0       | Not           |
| A         | 2024-03-30 | 0       | Not           |
| A         | 2024-03-31 | 0       | Not           |
| A         | 2024-04-01 | 1       | Easter Monday |
| A         | 2024-04-02 | 0       | Not           |
| A         | 2024-04-03 | 0       | Not           |

#### Output - 2 days before and 1 day after:

| warehouse | date       | holiday | holiday_name  |
|-----------|------------|---------|---------------|
| A         | 2024-03-28 | 0       | Not           |
| A         | 2024-03-29 | 0       | Not           |
| A         | 2024-03-30 | 1.3     | Easter Monday |
| A         | 2024-03-31 | 1       | Easter Monday |
| A         | 2024-04-01 | 1       | Easter Monday |
| A         | 2024-04-02 | 1       | Not           | 
| A         | 2024-04-03 | 0       | Not           |

#### I not fill holiday name for day after holiday

In [ ]:
# Function to fill holidays from calendar to datafarme
# Main idea:
# 
def fill_calendar2df(df, df_cld):
    # Convert holiday to 1 if it is None
    df['holiday'] = df['holiday'].fillna(0)
    hl_count = 0 # count total number of holidays
    spec = 0 # count number of special holidays
    
    # Iterate through each holiday in df_cld
    for _, row in df_cld.iterrows():
        warehouse = row['warehouse']
        holiday_date = row['date']
        holiday_name = row['holiday_name']

        is_spec = False
        hl_count += 1

        # Calculate the date range: 2 days before, the holiday itself, and 1 day after
        # we fill this date range as holiday
        if (holiday_name in ['Labour Day','Easter Monday']): # special holidays
            date_range = pd.date_range(start=holiday_date - pd.Timedelta(days=2), end=holiday_date + pd.Timedelta(days=1))
            is_spec = True
        else: # with other holidays: 1 days before, the holiday itself, and 1 day after
            date_range = pd.date_range(start=holiday_date - pd.Timedelta(days=1), end=holiday_date + pd.Timedelta(days=1))

        # Update df for the corresponding warehouse and date range
        for i, date in enumerate(date_range):
            mask = (df['warehouse'] == warehouse) & (df['date'] == date)
            if is_spec and i==0:
                spec += 1
                df.loc[mask, 'holiday'] = 1.3 # Use higher weight for 2 special holidays
            else:
                df.loc[mask, 'holiday'] = 1

            # not fill holiday name for day after holiday
            if i+1!=len(date_range):
                df.loc[mask, 'holiday_name'] = holiday_name

    print("Total holidays:", hl_count)
    print("Total special holidays:", spec)
    return df

# Reset state of df
df = df_c.copy()
test = test_c.copy()

# Manual fill all nan holiday name to "Not"
df = df.fillna('Not')
test = test.fillna('Not')

df['holiday'] = df['holiday'].astype(float)
test['holiday'] = test['holiday'].astype(float)

# Fill holidays from calender.csv to df
print('Dataframe train')
df = fill_calendar2df(df, df_cld)
print('Dataframe test')
test = fill_calendar2df(test, test_cld)


In [ ]:
# Day before "Easter Monday" list
datesx = ['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020']

# Convert the dates to datetime objects and subtract one day
holidaysx = [datetime.strptime(date, '%m/%d/%Y') - timedelta(days=1) for date in datesx]

# Define the list of warehouses
warehouses = ['Prague_1', 'Prague_2', 'Prague_3']

# Set 'holiday' to 1 for matching rows
df.loc[(df['date'].isin(holidaysx)) & (df['warehouse'].isin(warehouses)), 'holiday'] = 1
test.loc[(test['date'].isin(holidaysx)) & (test['warehouse'].isin(warehouses)), 'holiday'] = 1

## 3. Create 2 new features

In [ ]:
# Create 2 new columns "day_before_holiday" and "day_after_holiday"
df['day_before_holiday'] = df['holiday'].shift(-1).fillna(0)
df['day_after_holiday'] = df['holiday'].shift().fillna(0)
test['day_before_holiday'] = test['holiday'].shift(-1).fillna(0)
test['day_after_holiday'] = test['holiday'].shift().fillna(0)
df['day_before_holiday'] = df['day_before_holiday'].astype(int)
df['day_after_holiday'] = df['day_after_holiday'].astype(int)
test['day_before_holiday'] = test['day_before_holiday'].astype(int)
test['day_after_holiday'] = test['day_after_holiday'].astype(int)

## 4. Main data processing function

### **Features Engineering with OnehotEncoder and TFIDF and data_processing**

In [ ]:
# Data processing function
def data_process(df:pd.DataFrame, create_test_mode:bool, target_cl:list, test_target_cl:list=[], onehot_encoder=None, tfidfvectorizer=None):
    df = df[df['warehouse'].isin(target_cl)]

    if not test_target_cl:
        test_target_cl = target_cl
    testids = test['id']

    # drop columns
    ignore_columns = ['id','shutdown','mini_shutdown', 'blackout',
                      'mov_change', 'frankfurt_shutdown', 'precipitation', 'snow',
                      'user_activity_1', 'user_activity_2',
                      ]
    df = df.drop(ignore_columns, axis=1, errors="ignore")

    # Create new columns
    DATE_COLUMNS = ['date']
    for _col in DATE_COLUMNS:
        train_date_col = pd.to_datetime(df[_col], errors='coerce')
        df[_col + "_year"] = train_date_col.dt.year.fillna(-1)
        df[_col + "_month"] = train_date_col.dt.month.fillna(-1)
        df[_col + "_day"] = train_date_col.dt.day.fillna(-1)
        df[_col + "_day_of_week"] = train_date_col.dt.dayofweek.fillna(-1)

        # df['quarter'] = df['date'].dt.quarter
        # df['season'] = (df[_col + "_month"] % 12 + 3) // 3
        # df['season'] = df['season'].astype(int)

        df.drop(_col, axis=1, inplace=True)

    # processing holiday_name
    TEXT_COLUMNS = ['holiday_name']
    def process_text(__dataset):
        for _col in TEXT_COLUMNS:
            process_text = [t.lower() for t in __dataset[_col]]
            # strip all punctuation
            table = str.maketrans('', '', string.punctuation)
            process_text = [t.translate(table) for t in process_text]
            # convert all numbers in text to 'num'
            process_text = [re.sub(r'\d+', 'num', t) for t in process_text]
            __dataset[_col] = process_text
        return __dataset
    df = process_text(df)
    
    
    TARGET_COLUMNS = ['orders']
    if set(TARGET_COLUMNS).issubset(df.columns.tolist()):
        feature_train = df.drop(TARGET_COLUMNS, axis=1)
        target_train = df[TARGET_COLUMNS].copy()
    else:
        feature_train = df

    # Onehot
    CATEGORICAL_COLS = ['warehouse']

    if len(target_cl) != 1:
        if not create_test_mode:
            train_encoded = pd.DataFrame(onehot_encoder.fit_transform(feature_train[CATEGORICAL_COLS]), columns=onehot_encoder.get_feature_names_out(), index=feature_train.index)
        else:
            train_encoded = pd.DataFrame(onehot_encoder.transform(feature_train[CATEGORICAL_COLS]), columns=onehot_encoder.get_feature_names_out(), index=feature_train.index)
        feature_train = pd.concat([feature_train, train_encoded ], axis=1)

    feature_train.drop(CATEGORICAL_COLS, axis=1, inplace=True)

    TEXT_COLUMNS = ['holiday_name']
    temp_train_data = feature_train[TEXT_COLUMNS]
    # Make the entire dataframe sparse to avoid it converting into a dense matrix.
    feature_train = feature_train.drop(TEXT_COLUMNS, axis=1).astype(pd.SparseDtype('float64', 0))

    for _col in TEXT_COLUMNS:
        if not create_test_mode:
            vector_train = tfidfvectorizer.fit_transform(temp_train_data[_col])
        else:
            vector_train = tfidfvectorizer.transform(temp_train_data[_col])
        feature_names = ['_'.join([_col, name]) for name in tfidfvectorizer.get_feature_names_out()]
        vector_train = pd.DataFrame.sparse.from_spmatrix(vector_train, columns=feature_names, index=temp_train_data.index)
        feature_train = pd.concat([feature_train, vector_train], axis=1)

    if create_test_mode:
        return feature_train, testids
    else:
        print(feature_names)
        return feature_train, target_train, onehot_encoder, tfidfvectorizer

In [ ]:
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
tfidfvectorizer = TfidfVectorizer(max_features=3000)
feature_train, target_train, onehot_encoder, tfidfvectorizer = data_process(df, create_test_mode=False,
                                                                  target_cl=list(df.warehouse.unique()),
                                                                  onehot_encoder=onehot_encoder, tfidfvectorizer=tfidfvectorizer)

feature_test, testids = data_process(test, create_test_mode=True,
                                        target_cl=list(df.warehouse.unique()),
                                        onehot_encoder=onehot_encoder, tfidfvectorizer=tfidfvectorizer)


### **Represents training information and testing information.**

In [ ]:
feature_train

In [ ]:
target_train

In [ ]:
feature_test

In [ ]:
testids

# 4. Loss function and predictor

## **The loss function we use is MAE and we use MAPE to evaluate**

In [ ]:
#evaluate
def MAPE(y_true,y_pred):
    return np.mean(np.abs(y_pred-y_true)/y_true)

def pred_test(algo, X_train, y_train, X_test, exp_mode=False, **kwarg):
    if not exp_mode:
        model = algo.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        # Training on log features, help model more stable
        model = algo.fit(X_train, np.log1p(y_train))
        y_pred = np.expm1(model.predict(X_test))

    return y_pred, model

def save_submit(y_pred, save_dir='/kaggle/working'):
    TARGET_COLUMNS = ['orders']
    prediction = pd.DataFrame(y_pred, columns=TARGET_COLUMNS)

    merged_prediction = pd.concat([testids, prediction], axis=1)
    merged_prediction = merged_prediction.reset_index(drop=True)
    merged_prediction.to_csv(f"{save_dir}/submission.csv",index=False)
    return merged_prediction
def fit_ml_algo(algo, X_train, y_train, X_test, y_test, exp_mode=False):
    if not exp_mode:
        model = algo.fit(X_train, y_train)

        y_pred = model.predict(X_test)
    else:
        model = algo.fit(X_train, np.log1p(y_train))
        y_pred = np.expm1(model.predict(y_train))
    mape = MAPE(y_test,y_pred)
    print("MAPE:", mape)

    return y_pred, model


# 5. Training and predict

In [ ]:
# hyperparameter tune by optuna
best_hypr = {'reg_alpha': 0.22395576225297806,
             'reg_lambda': 0.013055491064310818,
             'learning_rate': 0.48284825276236043,
             'colsample_bytree': 0.7922000123536603,
             'min_child_weight': 0.00010297333065138669,
             'num_leaves': 14,
             'min_child_samples': 10}

# model = LGBMRegressor(importance_type='gain', min_child_samples=9, n_estimators=100, force_col_wise=True)
model = LGBMRegressor(objective='regression_l1', n_estimators=600, **best_hypr)
y_pred, model = pred_test(model, feature_train, target_train.values.ravel(), feature_test, exp_mode=True)

print(y_pred.mean())

y_pred_self = np.expm1(model.predict(feature_train))
print('Self fit loss:', MAPE(target_train.values.ravel(), y_pred_self))

submit_df = save_submit(y_pred, save_dir="/kaggle/working/")
submit_df_c = submit_df.copy()
df.to_csv(f'{checkpoint_path}/df_temp.csv')
test.to_csv(f'{checkpoint_path}/test_temp.csv')
# 0.021298697136921178

# 6. Show some plots

## 1. Most important features

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(10, 20))

palette = sns.color_palette("RdYlGn_r", len(feature_train.columns))
lgbm_importances = pd.Series(model.feature_importances_, index=feature_train.columns).sort_values(ascending=False)
sns.barplot(y=lgbm_importances.index, x=lgbm_importances.values, orient='h', palette=palette)

plt.tight_layout()
plt.show()

## 2. End predict

In [ ]:
# Show train data compare train predict and test predict
df = df_ori.copy()
df['y_pred_self'] = y_pred_self

df.set_index('date', inplace=True)

# Extract 'warehouse' and 'date' from 'id'
submit_df = submit_df_c.copy()
submit_df['warehouse'] = submit_df['id'].apply(lambda x: x.rsplit('_', maxsplit=1)[0])
submit_df['date'] = submit_df['id'].apply(lambda x: x.split('_')[-1])
submit_df['date'] = pd.to_datetime(submit_df['date'])

submit_df.set_index('date', inplace=True)

for warehouse in df['warehouse'].unique():

    df_warehouse = df[df['warehouse'] == warehouse]
    submit_warehouse = submit_df[submit_df['warehouse'] == warehouse]

    plt.figure(figsize=(14, 6))

    # True orders from the training data
    plt.plot(df_warehouse.index, df_warehouse['orders'], label='True Orders', linewidth=0.8)

    # Predicted orders of the training data
    plt.plot(df_warehouse.index, df_warehouse['y_pred_self'], label='Predict Orders (Train)', linestyle='--', linewidth=0.8)

    # Predicted orders of the test set
    plt.plot(submit_warehouse.index, submit_warehouse['orders'], label='Predict Orders (Test)', linestyle='-', linewidth=0.8)

    plt.title(f'Orders vs Predicted Orders for Warehouse: {warehouse}')
    plt.xlabel('Date')
    plt.ylabel('Orders')
    plt.legend()
    plt.show()